In [5]:
import pandas as pd
newer = pd.read_csv("MPS_Borough_Level_Crime _Most_Recent_24_months_new.csv")
print(newer.columns.tolist()[-6:])

['202601', '202602', '202603', '202604', '202605', '202606']


In [6]:
newer = pd.read_csv("MPS_Borough_Level_Crime_Most recent_July_month_validation.csv")  # or your new filename
print(newer.columns.tolist()[-6:])

['202602', '202603', '202604', '202605', '202606', '202607']


In [7]:
jul_col = "202607"

london_actual_jul = newer[jul_col].sum()
print("London-wide actual, July 2026:", london_actual_jul)

borough_actual_jul = newer.groupby("BOCU")[jul_col].sum().reset_index()
borough_actual_jul.columns = ["Borough", "Actual_2026-07"]
borough_actual_jul = borough_actual_jul[borough_actual_jul["Borough"] != "Unknown"]
print(borough_actual_jul.shape)
borough_actual_jul.head()

London-wide actual, July 2026: 83665
(33, 2)


,Borough,Actual_2026-07
0,Aviation Policing,1
1,Barking and Dagenham,1874
2,Barnet,2493
3,Bexley,1538
4,Brent,2935


In [8]:
borough_actual_jul = borough_actual_jul[borough_actual_jul["Borough"] != "Aviation Policing"]
print(borough_actual_jul.shape)  # should now be 32

(32, 2)


In [9]:
jul_forecast = pd.read_csv("borough_future_forecasts_Jul-Dec_2026.csv")
print(jul_forecast.columns.tolist())
print(jul_forecast.head())

['Borough', 'Month', 'Forecast', 'Lower_95CI', 'Upper_95CI']
                Borough    Month  Forecast  Lower_95CI  Upper_95CI
0  Barking and Dagenham  2026-07      1930        1709        2152
1  Barking and Dagenham  2026-08      1810        1536        2085
2  Barking and Dagenham  2026-09      1872        1545        2200
3  Barking and Dagenham  2026-10      1975        1604        2346
4  Barking and Dagenham  2026-11      1968        1558        2378


In [10]:
# Filter to just July forecasts
jul_forecast_only = jul_forecast[jul_forecast["Month"] == "2026-07"].copy()

# Merge with real July actuals
jul_comparison = jul_forecast_only.merge(borough_actual_jul, on="Borough")

# Calculate error
jul_comparison["Pct_Error"] = (abs(jul_comparison["Actual_2026-07"] - jul_comparison["Forecast"]) / jul_comparison["Actual_2026-07"] * 100).round(1)

# Check if actual fell within the 95% CI
jul_comparison["Within_CI"] = (jul_comparison["Actual_2026-07"] >= jul_comparison["Lower_95CI"]) & (jul_comparison["Actual_2026-07"] <= jul_comparison["Upper_95CI"])

jul_comparison = jul_comparison.sort_values("Pct_Error")

print("Mean error across 32 boroughs:", round(jul_comparison["Pct_Error"].mean(), 2), "%")
print("Boroughs where actual fell within 95% CI:", jul_comparison["Within_CI"].sum(), "/ 32")
print()
print(jul_comparison[["Borough", "Forecast", "Actual_2026-07", "Pct_Error", "Within_CI"]])

# Also check London-wide total
london_forecast_jul = jul_forecast_only["Forecast"].sum()
london_pct_error = round(abs(london_actual_jul - london_forecast_jul) / london_actual_jul * 100, 2)
print(f"\nLondon-wide: Forecast={london_forecast_jul}, Actual={london_actual_jul}, Error={london_pct_error}%")

Mean error across 32 boroughs: 4.54 %
Boroughs where actual fell within 95% CI: 32 / 32

                   Borough  Forecast  Actual_2026-07  Pct_Error  Within_CI
14                Havering      1922            1914        0.4       True
19    Kingston upon Thames      1139            1134        0.4       True
2                   Bexley      1547            1538        0.6       True
6                  Croydon      3349            3312        1.1       True
11  Hammersmith and Fulham      1917            1890        1.4       True
21                Lewisham      3028            3076        1.6       True
7                   Ealing      3130            3072        1.9       True
23                  Newham      3834            3759        2.0       True
20                 Lambeth      3755            3848        2.4       True
13                  Harrow      1587            1547        2.6       True
18  Kensington and Chelsea      1741            1788        2.6       True
4          

In [11]:
jul_comparison.to_csv("data/july_2026_validation_all32.csv", index=False)
print("Saved!")

Saved!
